In [44]:
import sqlite3
import pandas as pd
# connect database
conn = sqlite3.connect(r"C:\Users\hamen\OneDrive\Desktop\project\database\project1.db")   # replace with your DB file
# Run SQL query and directly load into DataFrame with columns
query = """select*from customers"""
df = pd.read_sql_query(query,conn)
df



,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle-aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle-aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young Adult,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle-aged,365
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3895,3896,40,Female,Hoodie,Clothing,28,Virginia,L,Turquoise,Summer,4.2,No,2-Day Shipping,No,32,Venmo,Weekly,Adult,7
3896,3897,52,Female,Backpack,Accessories,49,Iowa,L,White,Spring,4.5,No,Store Pickup,No,41,Bank Transfer,Bi-Weekly,Middle-aged,14
3897,3898,46,Female,Belt,Accessories,33,New Jersey,L,Green,Spring,2.9,No,Standard,No,24,Venmo,Quarterly,Middle-aged,90
3898,3899,44,Female,Shoes,Footwear,77,Minnesota,S,Brown,Summer,3.8,No,Express,No,24,Venmo,Weekly,Adult,7


Total revenue generated by both male and female customers

In [45]:

query= """select gender,sum(purchase_amount) Total_revenue from customers group by gender order by Total_revenue desc"""
df = pd.read_sql_query(query,conn)
df


,gender,Total_revenue
0,Male,157890
1,Female,75191


which customers used a discount but still spend more than the average purchase amount

In [46]:
df = pd.read_sql_query("select customer_id,purchase_amount from customers where purchase_amount > (select avg(purchase_amount) from customers) and discount_applied ='Yes';",conn)

print(df)

     customer_id  purchase_amount
0              2               64
1              3               73
2              4               90
3              7               85
4              9               97
..           ...              ...
834         1667               64
835         1671               73
836         1673               73
837         1674               62
838         1676               90

[839 rows x 2 columns]


which are the top 5 products with the highest average review rating


In [47]:
query = """select item_purchased,round(avg(review_rating),2) from customers group by item_purchased ORDER by avg(review_rating) DESC LIMIT 5;"""
df = pd.read_sql_query(query,conn)
df

,item_purchased,"round(avg(review_rating),2)"
0,Gloves,3.86
1,Sandals,3.84
2,Boots,3.82
3,Hat,3.80
4,Skirt,3.78


compare the average purchase amounts between standard and express shipping

In [48]:
query = """SELECT shipping_type, AVG(purchase_amount) AS avg_purchase FROM customers WHERE shipping_type IN ("Express","Standard")GROUP BY shipping_type;"""
df = pd.read_sql_query(query,conn)
df

,shipping_type,avg_purchase
0,Express,60.475232
1,Standard,58.460245


Compare average spend and total revenue b/w subscirbers and non-subscribers

In [49]:
query = """select subscription_status,count(customer_id) as total_customers, round(avg(purchase_amount),2) as avg_spend, sum(purchase_amount) as total_reve from customers GROUP by subscription_status;"""
df = pd.read_sql_query(query,conn)
df

,subscription_status,total_customers,avg_spend,total_reve
0,No,2847,59.87,170436
1,Yes,1053,59.49,62645


which 5 products have the highest percentage of purchase with discount applied

In [50]:
query = """
select item_purchased,100*sum(case when discount_applied ='Yes' then 1 else 0 end)/count(*) as discount_rate from customers group by item_purchased order by discount_rate desc limit 5;"""
df = pd.read_sql_query(query,conn)
df

,item_purchased,discount_rate
0,Hat,50
1,Sneakers,49
2,Coat,49
3,Sweater,48
4,Pants,47


In [51]:
query ="""
with vt as (
select customer_id,previous_purchases,case 
when previous_purchases <10 then "New_cust"
when previous_purchases < 25 then "Repeated_cust"
else "Loyal_customer" end as customer_type from customers)
select customer_type, count(customer_id) as total_cust from vt group by customer_type;"""

df = pd.read_sql_query(query,conn)
df

,customer_type,total_cust
0,Loyal_customer,2014
1,New_cust,708
2,Repeated_cust,1178


what are the top 3 most purchased products within each category

In [52]:
query="""
with item_count as (
select category, item_purchased, count(customer_id) as total_orders,
row_number() over(PARTITION by category order by count(customer_id) desc) as item_rank from customers GROUP by category,item_purchased)
select category,item_rank,item_purchased,total_orders from item_count where item_rank <= 3;"""

df = pd.read_sql_query(query,conn)
df

,category,item_rank,item_purchased,total_orders
0,Accessories,1,Jewelry,171
1,Accessories,2,Sunglasses,161
2,Accessories,3,Belt,161
3,Clothing,1,Pants,171
4,Clothing,2,Blouse,171
5,Clothing,3,Shirt,169
6,Footwear,1,Sandals,160
7,Footwear,2,Shoes,150
8,Footwear,3,Sneakers,145
9,Outerwear,1,Jacket,163


Are customers who are repeat buyers (more than 5 previous purchase) also likey to subscribe?

In [53]:
query = """ 
select subscription_status,count(customer_id) as repeate_buyers from customers
group by subscription_status having previous_purchases > 5;"""
df = pd.read_sql_query(query,conn)
df

,subscription_status,repeate_buyers
0,No,2847
1,Yes,1053


what is the revenue contribution of each age groups?

In [54]:
query = """
select age_group,sum(purchase_amount)total_revenue from customers 
GROUP by age_group order by total_revenue desc;"""
df = pd.read_sql_query(query,conn)
df

,age_group,total_revenue
0,Young Adult,62143
1,Middle-aged,59197
2,Adult,55978
3,Senior,55763
